# Task 4 — WordNet similarities (WordSim353 sample)

Решение задания на основе датасета `Task_4_sample_15.csv`.

In [1]:
%pip install pandas scipy nltk

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import nltk
from nltk.corpus import wordnet as wn
from itertools import product
from scipy.stats import spearmanr

In [3]:
nltk.download("wordnet")

url = "https://storage.yandexcloud.net/lms-itmo-ru-files-27a87tyf/Text_Processing/Task_4/samples/Task_4_sample_15.csv"
df = pd.read_csv(url)
df.head()

[nltk_data] Downloading package wordnet to /Users/afafos/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,word_1,word_2,Score
0,lad,brother,4.46
1,doctor,nurse,7.00
2,cup,substance,1.92
3,drink,car,3.04
4,lad,wizard,0.92


In [4]:
path_scores_max = []
lch_scores_max = []
wup_scores_max = []
true_scores = []

for _, row in df.iterrows():
    word_1 = row["word_1"]
    word_2 = row["word_2"]
    score = row["Score"]

    synsets_1 = wn.synsets(word_1, pos="n")
    synsets_2 = wn.synsets(word_2, pos="n")

    path_scores = []
    lch_scores = []
    wup_scores = []

    for s1, s2 in product(synsets_1, synsets_2):
        path_value = s1.path_similarity(s2)
        if path_value is not None:
            path_scores.append(path_value)

        try:
            lch_value = s1.lch_similarity(s2)
            if lch_value is not None:
                lch_scores.append(lch_value)
        except Exception:
            pass

        wup_value = s1.wup_similarity(s2)
        if wup_value is not None:
            wup_scores.append(wup_value)

    path_scores_max.append(max(path_scores) if path_scores else 0)
    lch_scores_max.append(max(lch_scores) if lch_scores else 0)
    wup_scores_max.append(max(wup_scores) if wup_scores else 0)
    true_scores.append(score)

In [5]:
path_corr, _ = spearmanr(path_scores_max, true_scores)
lch_corr, _ = spearmanr(lch_scores_max, true_scores)
wup_corr, _ = spearmanr(wup_scores_max, true_scores)

round(path_corr, 3), round(lch_corr, 3), round(wup_corr, 3)

(np.float64(0.609), np.float64(0.609), np.float64(0.627))

In [6]:
seafood_synset = wn.synset("seafood.n.01")
seafood_hyponyms = sorted(seafood_synset.hyponyms(), key=lambda syn: syn.name())

len(seafood_hyponyms), seafood_hyponyms[0].name()

(11, 'freshwater_fish.n.01')

In [7]:
print(round(path_corr, 3))
print(round(lch_corr, 3))
print(round(wup_corr, 3))
print(len(seafood_hyponyms))
print(seafood_hyponyms[0].name())

0.609
0.609
0.627
11
freshwater_fish.n.01
